# Learning module 11 — Create our first iOS automation session

**Small milestone:** connect the Python client to the local Appium server and request an XCUITest session for our installed Movies app. Read the real returned session ID/capabilities, then close the session and stop our server. We will not search, tap, or evaluate the Interstellar scenario yet.

```mermaid
flowchart LR
    Python["Python: webdriver.Remote"] --> Appium["Appium server"]
    Appium --> Driver["XCUITest driver"]
    Driver --> WDA["WebDriverAgent / XCTest"]
    WDA --> App["Movies on the selected Simulator"]
```

In module 9, `XCUITestOptions` only prepared a request. **`webdriver.Remote(...)` sends that request.** Appium selects XCUITest, prepares the specified Simulator, builds/starts or reuses WebDriverAgent as appropriate, and requests a device automation session. A successful response gives the Python driver a session ID and returned capabilities. [Appium Python client](https://github.com/appium/python-client), [XCUITest capabilities](https://appium.github.io/appium-xcuitest-driver/latest/reference/capabilities/)

This step can install/start WebDriverAgent and launch or reuse Movies. Our existing `noReset=True` / `fullReset=False` settings request preservation of application data; they do not freeze UI state or prevent normal backend requests when the app runs. With this reset policy, the driver's default session-end behavior does not terminate the app. Closing automation is different from uninstalling Movies. [App lifecycle capabilities](https://appium.github.io/appium-xcuitest-driver/latest/reference/capabilities/#app)

**Before running:** select the project's `.venv` kernel. Run this notebook's setup cells before the target check and session cell. Module 10 deliberately stopped its server; this exercise starts and owns a new one. The earlier dependency findings remain unresolved; this exercise uses the trusted installed bundle, not an app download/archive or externally supplied filesystem patterns.

### First, revalidate our selected target

The next cell reads current Simulator metadata and checks the installed bundle's container. It requires the recorded Simulator to already be booted; it does not switch devices, boot one, reset one, or install Movies. These checks establish target availability, not successful automation.


## This notebook

Part **11 — Create and close an iOS session** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** May launch/reuse Movies; no scenario action.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.

### Setup for this notebook

These source-only prerequisite cells are repeated intentionally so the lesson is runnable in a fresh kernel. They configure local values/environment and inspect tool versions; they do not launch an Appium server, create a session, or tap/type in the app. The original lessons remain linked for explanation.


In [1]:
# Setup: Sample scenario input (lesson 01)
from pprint import pprint

scenario_text = (
    "In Movies, search for Interstellar, open the 2014 movie, "
    "and verify that its details show the title Interstellar "
    "and release year 2014."
)

scenario_input = {
    "instruction": scenario_text,
    "application_name": "Movies",
    "application_profile_ref": None,
    "requested_actions": [
        "search_for_movie",
        "open_matching_movie_details",
    ],
    "search_query": "Interstellar",
    "target_item_identity": {"title": "Interstellar", "release_year": 2014},
    "expected_title": "Interstellar",
    "expected_release_year": 2014,
    "title_matching_rule": "exact",  # Proposed comparison rule for this exercise.
}

pprint(scenario_input, sort_dicts=False)


{'instruction': 'In Movies, search for Interstellar, open the 2014 movie, and '
                'verify that its details show the title Interstellar and '
                'release year 2014.',
 'application_name': 'Movies',
 'application_profile_ref': None,
 'requested_actions': ['search_for_movie', 'open_matching_movie_details'],
 'search_query': 'Interstellar',
 'target_item_identity': {'title': 'Interstellar', 'release_year': 2014},
 'expected_title': 'Interstellar',
 'expected_release_year': 2014,
 'title_matching_rule': 'exact'}


In [2]:
# Setup: Recorded run profile (lesson 03)
from pprint import pprint

run_profile = {
    "id": "movies-ios-local",
    "platform": "ios",
    "execution_provider": "local",
    "app_bundle_id": "com.example.movies",
    "simulator_udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
}

app_profiles = {run_profile["id"]: run_profile}
scenario_input["application_profile_ref"] = run_profile["id"]
selected_profile = app_profiles[scenario_input["application_profile_ref"]]

print("Scenario profile reference:", scenario_input["application_profile_ref"])
pprint(selected_profile, sort_dicts=False)


Scenario profile reference: movies-ios-local
{'id': 'movies-ios-local',
 'platform': 'ios',
 'execution_provider': 'local',
 'app_bundle_id': 'com.example.movies',
 'simulator_udid': '4F0707AE-45BC-464D-9099-6EE76C3740A9'}


In [3]:
# Setup: Local Node activation (lesson 05)
import os
from pathlib import Path
import shutil
import subprocess

notebook_cwd = Path.cwd().resolve()
server_candidates = (notebook_cwd, notebook_cwd / "server", *notebook_cwd.parents)
server_dir = next((candidate for candidate in server_candidates
                   if (candidate / ".node-version").is_file()
                   and (candidate / "package.json").is_file()
                   and (candidate / "sample_scenarios").is_dir()), None)
if server_dir is None:
    raise RuntimeError("Run from the project root, server, or sample_scenarios folder.")

node_version = (server_dir / ".node-version").read_text().strip()
node_bin = server_dir / ".tools" / f"node-v{node_version}-darwin-arm64" / "bin"
if not (node_bin / "node").is_file():
    raise FileNotFoundError(f"Project-local Node is missing: {node_bin}")

path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(node_bin)]
os.environ["PATH"] = os.pathsep.join([str(node_bin), *path_entries])

print("Selected Node:", shutil.which("node"))
print("Node version:", subprocess.check_output(["node", "--version"], text=True, timeout=10).strip())
print("npm version:", subprocess.check_output(["npm", "--version"], text=True, timeout=10).strip())


Selected Node: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/node-v24.22.2-darwin-arm64/bin/node
Node version: v24.22.2
npm version: 11.19.1


In [4]:
# Setup: Local Appium activation (lesson 06)
import json
import os
import shutil
import subprocess

if "server_dir" not in globals() or "node_bin" not in globals():
    raise RuntimeError("Run the Node setup cell in this notebook first.")
if shutil.which("node") != str(node_bin / "node"):
    raise RuntimeError("Reactivate the project-local Node runtime first.")

appium_bin_dir = server_dir / "node_modules" / ".bin"
appium_executable = appium_bin_dir / "appium"
if not appium_executable.is_file():
    raise FileNotFoundError("The project-local Appium package is missing.")

os.environ["APPIUM_HOME"] = str(server_dir / ".tools" / "appium")
os.environ["npm_config_cache"] = str(server_dir / ".tools" / "npm-cache")
path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(appium_bin_dir)]
os.environ["PATH"] = os.pathsep.join([str(appium_bin_dir), *path_entries])

expected_version = json.loads((server_dir / "package.json").read_text())["devDependencies"]["appium"]
actual_version = subprocess.check_output(
    [str(appium_executable), "--version"], cwd=server_dir, text=True, timeout=30
).strip()
if actual_version != expected_version:
    raise RuntimeError(f"Expected Appium {expected_version}, found {actual_version}.")

print("Selected Appium:", shutil.which("appium"))
print("Appium version:", actual_version)
print("Appium home:", os.environ["APPIUM_HOME"])


Selected Appium: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/node_modules/.bin/appium
Appium version: 3.9.0
Appium home: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium


dbug Appium Creating hash file directory: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium/node_modules/.cache/appium


In [5]:
# Setup: Local iOS options (lesson 09)
import json
from appium.options.ios import XCUITestOptions

if "selected_profile" not in globals():
    raise RuntimeError("Run the scenario/profile setup cells in this notebook first.")
if selected_profile.get("platform") != "ios":
    raise ValueError("This example requires an iOS run profile.")
for field in ["app_bundle_id", "simulator_udid"]:
    value = selected_profile.get(field)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"The selected profile needs a nonempty {field}.")

ios_options = XCUITestOptions()
ios_options.udid = selected_profile["simulator_udid"]
ios_options.bundle_id = selected_profile["app_bundle_id"]
ios_options.no_reset = True
ios_options.full_reset = False

session_request = ios_options.to_w3c()
print(json.dumps(session_request, indent=2))


{
  "capabilities": {
    "firstMatch": [
      {}
    ],
    "alwaysMatch": {
      "appium:automationName": "XCUITest",
      "platformName": "iOS",
      "appium:udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
      "appium:bundleId": "com.example.movies",
      "appium:noReset": true,
      "appium:fullReset": false
    }
  }
}


In [6]:
import json
from pathlib import Path
import subprocess

if "selected_profile" not in globals() or "ios_options" not in globals():
    raise RuntimeError("Run the scenario/profile/options setup cells in this notebook first.")
if selected_profile.get("platform") != "ios" or selected_profile.get("execution_provider") != "local":
    raise ValueError("This exercise requires a local iOS profile.")
udid = selected_profile["simulator_udid"]
bundle_id = selected_profile["app_bundle_id"]
if ios_options.udid != udid or ios_options.bundle_id != bundle_id:
    raise ValueError("Profile and options differ; rerun this notebook's profile/options setup cells before continuing.")

devices = json.loads(subprocess.check_output(
    ["xcrun", "simctl", "list", "devices", "--json"], text=True, timeout=20,
))
matches = [(runtime, device) for runtime, entries in devices["devices"].items()
           for device in entries if device["udid"] == udid]
if len(matches) != 1 or not matches[0][1].get("isAvailable") or matches[0][1]["state"] != "Booted":
    raise RuntimeError("The selected Simulator must be available and booted; no alternate device was selected.")
runtime, device = matches[0]
app_path = subprocess.check_output(
    ["xcrun", "simctl", "get_app_container", udid, bundle_id, "app"],
    text=True, timeout=20,
).strip()
if not Path(app_path).is_dir():
    raise RuntimeError("The installed application container could not be verified.")
print(json.dumps({"device": device["name"], "udid": udid, "runtime": runtime,
                  "state": device["state"], "installed_bundle": bundle_id}, indent=2))


{
  "device": "iPhone 18 Pro Max",
  "udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
  "runtime": "com.apple.CoreSimulator.SimRuntime.iOS-27-0",
  "state": "Booted",
  "installed_bundle": "com.example.movies"
}


### Then, start Appium and request the session

Most of the server lifecycle code below repeats module 10 so this exercise owns its server and guarantees cleanup. The new connection is:

```python
client_config = AppiumClientConfig(remote_server_addr=server_url, timeout=300)
driver = webdriver.Remote(
    command_executor=server_url,
    options=connection_options,
    client_config=client_config,
)
```

`timeout=300` is the Python HTTP transport timeout, in **seconds**. It gives the initial Xcode/WDA setup time to respond; it is not the scenario timeout, a pass condition, or the driver's WDA timeout. We do not automatically retry a failed session request. [Client timeout configuration](https://github.com/appium/python-client#relaxing-http-request-read-timeout)

We copy module 9's options, leaving that original object unchanged. The connection copy adds an explicit loopback WDA binding and port `8100`. `4723` (Appium) must be free, so this exercise owns its Appium server. XCUITest manages WDA startup/reuse on `8100`; WDA may remain from a previous session. Run these exercises serially on the selected Simulator, without a competing automation run. Appium's HTTP endpoint and WDA's device-automation endpoint are separate. [WDA configuration](https://appium.github.io/appium-xcuitest-driver/latest/reference/capabilities/#webdriveragent)

`driver.quit()` sends the session deletion request. A nested `finally` stops our owned server even if session cleanup raises. A failed session constructor may not return a driver object; inspect the captured log rather than invent a session ID or a successful device result.


In [7]:
import json
import os
import shutil
import socket
import subprocess
import tempfile
import time
import urllib.error
import urllib.request
from appium import webdriver
from appium.options.ios import XCUITestOptions
from appium.webdriver.client_config import AppiumClientConfig
from selenium.webdriver.common.proxy import Proxy

if not all(name in globals() for name in ("server_dir", "node_bin", "appium_executable", "ios_options")):
    raise RuntimeError("Run this notebook's setup cells first (and its definitions cell where present).")
if shutil.which("node") != str(node_bin / "node"):
    raise RuntimeError("Reactivate the project-local Node runtime.")
if os.environ.get("APPIUM_HOME") != str(server_dir / ".tools" / "appium"):
    raise RuntimeError("Reactivate the project-local Appium home.")
if ios_options.no_reset is not True or ios_options.full_reset is not False:
    raise ValueError("Rerun this notebook's profile/options setup cells to restore this exercise's explicit reset policy.")

host, port, wda_port = "127.0.0.1", 4723, 8100
with socket.socket() as probe:
    probe.bind((host, port))  # Leave an existing Appium server alone.

connection_options = XCUITestOptions().load_capabilities(ios_options.to_capabilities())
connection_options.set_capability("appium:wdaBindingIP", host)
connection_options.set_capability("appium:wdaLocalPort", wda_port)
server_url = f"http://{host}:{port}"
client_config = AppiumClientConfig(
    remote_server_addr=server_url, timeout=300,
    proxy=Proxy({"proxyType": "direct"}),
    init_args_for_pool_manager={"init_args_for_pool_manager": {"retries": 0}},
)
expected_version = json.loads((server_dir / "package.json").read_text())["devDependencies"]["appium"]
log_dir = server_dir / ".tools" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(mode="w", prefix="session-config-", suffix=".json",
                                 dir=log_dir, delete=False) as config_file:
    json.dump({"server": {}}, config_file)
status_opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
session_driver, session_process, session_result = None, None, None
with tempfile.NamedTemporaryFile(mode="w", prefix="session-check-", suffix=".log",
                                 dir=log_dir, delete=False) as session_log:
    print("Server/session log:", session_log.name, flush=True)
    try:
        session_process = subprocess.Popen(
            [str(node_bin / "node"), str(appium_executable), "--config", config_file.name,
             "--address", host, "--port", str(port), "--log-no-colors", "--log-timestamp"],
            cwd=server_dir, env=os.environ.copy(), stdout=session_log, stderr=subprocess.STDOUT,
        )
        print("Owned Appium PID:", session_process.pid, flush=True)
        deadline = time.monotonic() + 30
        while time.monotonic() < deadline:
            if session_process.poll() is not None:
                raise RuntimeError(f"Appium exited early; inspect {session_log.name}")
            try:
                with status_opener.open(server_url + "/status", timeout=2) as response:
                    server_status = json.load(response)["value"]
                break
            except urllib.error.URLError:
                time.sleep(0.25)
        else:
            raise TimeoutError(f"Appium startup timed out; inspect {session_log.name}")
        if server_status.get("ready") is not True or server_status.get("build", {}).get("version") != expected_version:
            raise RuntimeError("Appium did not return the expected readiness/version.")
        if session_process.poll() is not None:
            raise RuntimeError("The owned server stopped before session creation.")
        print("Requesting XCUITest session; first WDA setup may take time...", flush=True)
        session_driver = webdriver.Remote(
            command_executor=server_url, options=connection_options, client_config=client_config,
        )
        if not session_driver.session_id:
            raise RuntimeError("The client did not receive a session ID.")
        session_result = {"session_id": session_driver.session_id,
                          "returned_capabilities": session_driver.capabilities}
        print(json.dumps(session_result, indent=2), flush=True)
    finally:
        try:
            if session_driver is not None:
                client_config.timeout = 20  # Bound cleanup requests separately from startup.
                session_driver.quit()
                print("Automation session closed.", flush=True)
        finally:
            if session_process is not None:
                if session_process.poll() is None:
                    session_process.terminate()
                    try:
                        session_process.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        session_process.kill()
                        session_process.wait(timeout=5)
                print("Owned Appium process stopped:", session_process.poll() is not None, flush=True)


Server/session log: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/logs/session-check-rz3130ut.log
Owned Appium PID: 5322
Requesting XCUITest session; first WDA setup may take time...
{
  "session_id": "6e42e379-f83c-4b11-af06-83218efe9a3e",
  "returned_capabilities": {
    "webStorageEnabled": false,
    "locationContextEnabled": false,
    "browserName": "",
    "platform": "MAC",
    "javascriptEnabled": true,
    "databaseEnabled": false,
    "takesScreenshot": true,
    "networkConnectionEnabled": false,
    "platformName": "iOS",
    "automationName": "XCUITest",
    "udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
    "bundleId": "com.example.movies",
    "noReset": true,
    "fullReset": false,
    "wdaBindingIP": "127.0.0.1",
    "wdaLocalPort": 8100,
    "platformVersion": "27.0"
  }
}
Automation session closed.
Owned Appium process stopped: True


### Read the session result, then pause

The returned capabilities may contain legacy fields such as `platform: MAC` alongside `platformName: iOS`. Read `platformName`, the selected UDID, and `platformVersion` for this target; do not interpret that legacy field as a macOS app test. Likewise, `takesScreenshot: true` advertises a capability; it is not a captured screenshot.

A returned session ID means the device automation session was created. Returned capabilities describe that session's negotiated configuration; echoed settings are not proof of a successful search, correct UI content, or a fresh backend call.

| Evidence | What it establishes |
| --- | --- |
| Current Simulator/container check | The selected target was booted and the Movies bundle was installed. |
| Appium `/status` response | Our HTTP server was responding before the request. |
| Real returned session ID/capabilities | The server accepted and completed device-session creation. |
| Session deletion response and server exit | The automation connection and owned server were cleaned up. |

The output is a historical snapshot. `session_result` retains the returned metadata for learning, but the closed `session_driver` must not be used for later actions. Each later exercise must own or explicitly reuse a still-live session. WDA build/install artifacts and reusable WDA processes can remain, and Movies may remain running; this is not an uninstall or restoration of the previous UI state.

If creation fails, the exception and local log are evidence of setup failure. Do not treat it as an Interstellar test failure: we have not executed that scenario. A successful session also does not establish the application's real API integration or clear the earlier dependency findings.

**Pause here:** trace requested capabilities → `POST /session` → device session ID → `DELETE /session/{id}`. Our next small milestone will be observing the current UI hierarchy and capturing a screenshot, before choosing any locator or tap. No LLM, classifier, LangGraph node, or agent has been introduced by this setup exercise.


**Recorded verification:** `2026-11-02T21:25:15-06:00`. The session returned `platformName: iOS`, `platformVersion: 27.0`, the selected UDID, and `bundleId: com.example.movies`. Its session ID is shown in the saved output. The captured log confirms HTTP 200 for both creation and deletion, followed by Appium server shutdown with exit code `0`. The final exercise was also checked after the initial WDA build so it permits the driver's normal reuse behavior.

Full local log: `/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/logs/session-check-s9bhkxm2.log` (ignored artifact). The notebook stores this metadata/output, not a still-active session.


[Index](README.md) · [Previous lesson](10_appium_server_status.ipynb) · [Next lesson](12_observe_ui.ipynb)